In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r'C:\Users\devpa\OneDrive\Desktop\Backup\Jarvis_V2\jarvis_intents.csv')
print( df.sample(10))

                      command           intent
29   Tell me the updated time         get_time
123  I want to use Calculator  open_calculator
125  Can you open Calculator?  open_calculator
99        Open Notepad editor     open_notepad
120           Open Calculator  open_calculator
249    Launch Spotify program     open_spotify
142    Start basic calculator  open_calculator
225                Run GitHub      open_github
184     Open LinkedIn website    open_linkedin
23   Jarvis, what's the time?         get_time


In [3]:
df['intent'].value_counts()

intent
get_time           30
search_google      30
search_youtube     30
open_notepad       30
open_calculator    30
open_whatsapp      30
open_linkedin      30
open_github        30
open_spotify       30
exit               30
Name: count, dtype: int64

In [4]:
df.describe()


,command,intent
count,300,300
unique,300,10
top,Goodbye assistant,get_time
freq,1,30


In [5]:
df['intent_number'] = df['intent'].map({
    'get_time': 0,
    'search_google': 1,
    'search_youtube': 2,
    'open_notepad': 3,
    'open_calculator': 4,
    'open_whatsapp': 5,
    'open_linkedin': 6,
    'open_github': 7,
    'open_spotify': 8,
    'exit': 9
})
print( df.sample(10))

                                      command           intent  intent_number
38          Search for stock prices on Google    search_google              1
149               Open calculator immediately  open_calculator              4
45                       Google latest movies    search_google              1
279                        Stop the assistant             exit              9
107                     Open Notepad software     open_notepad              3
204                 Launch LinkedIn instantly    open_linkedin              6
161                  Start WhatsApp messenger    open_whatsapp              5
221                          Open GitHub page      open_github              7
74   Search for educational videos on YouTube   search_youtube              2
188                  Open my LinkedIn profile    open_linkedin              6


In [6]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

stopwords = list(STOP_WORDS)
print(stopwords)


['have', 'why', 'if', 'next', 'back', 'by', 'name', 'who', 'therein', 'had', 'keep', 'well', 'seeming', 'were', 'them', 'bottom', 'besides', 'hereupon', 'give', 'though', 'often', 'whatever', 'around', 'him', 'made', 'cannot', 'nothing', 'somewhere', 'last', 'whose', '‘m', 'until', 'thru', 'seemed', 'seem', 'side', 'ourselves', 'whereby', '’ll', '‘ll', '’s', 'since', 'least', 'where', 'into', 'ours', 'thus', 'are', 'might', 'itself', 'still', 'nevertheless', 'becomes', 'show', 'thereby', 'been', 'both', 'behind', 'does', '‘ve', 'all', 'whoever', 'either', 'towards', 'something', 'did', 'or', 'those', 'anything', 'never', 'indeed', 'your', 'very', 'otherwise', '‘d', 'none', 'get', 'five', 'top', 'everything', 'forty', 'at', 'someone', 'onto', 'become', 'whenever', 'nobody', 'three', '’ve', 'than', 'others', 'during', 'beside', 'me', 'not', 'above', 'whence', 'whether', 'these', 'you', 'am', 'just', 'four', 'this', 'n‘t', 'when', 'there', 'much', 'whereupon', 'herself', 'n’t', 'regarding

In [7]:
df['command_without_stopwords'] = df['command'].apply(lambda x: ' '.join([word for word in x.split() if word not in stopwords]))

In [8]:
df.head(10)

,command,intent,intent_number,command_without_stopwords
0,What time is it?,get_time,0,What time it?
1,Can you tell me the time?,get_time,0,Can tell time?
2,Current time please,get_time,0,Current time
3,Tell me the time,get_time,0,Tell time
4,What’s the current hour?,get_time,0,What’s current hour?
5,Time now,get_time,0,Time
6,Could you give me the time?,get_time,0,Could time?
7,Do you know the time?,get_time,0,Do know time?
8,Time check,get_time,0,Time check
9,Show me the current time,get_time,0,Show current time


In [9]:
from spacy.lang.en.stop_words import STOP_WORDS
stopwords = list(STOP_WORDS)
df['command_without_stopwords'] = df['command'].apply(
    lambda x: ' '.join([word for word in x.split() if word.lower() not in stopwords])
)

# 4️⃣ Convert to embeddings
nlp = spacy.load('en_core_web_md')
df['command_vector'] = df['command_without_stopwords'].apply(lambda x: nlp(x).vector)

In [10]:
df.sample(10)


,command,intent,intent_number,command_without_stopwords,command_vector
202,Start LinkedIn now,open_linkedin,6,Start LinkedIn,"[-0.885755, 0.73415995, -0.303245, -0.23432499..."
201,Check LinkedIn notifications,open_linkedin,6,Check LinkedIn notifications,"[-1.0022401, 0.40286598, -0.27994666, -0.26004..."
142,Start basic calculator,open_calculator,4,Start basic calculator,"[-0.67868334, 0.30931333, -0.35042667, 0.10083..."
36,Look up weather today,search_google,1,Look weather today,"[-0.7675867, 0.38898668, -0.12962334, -0.14626..."
12,Tell me the exact time,get_time,0,Tell exact time,"[-0.67525005, 0.15935, -0.37264657, -0.1790999..."
156,Can you open WhatsApp?,open_whatsapp,5,open WhatsApp?,"[-0.67768, -0.0064033368, 0.011849006, 0.01197..."
56,Google funny memes,search_google,1,Google funny memes,"[-0.7029166, 0.080230005, -0.197194, -0.118986..."
18,What does the clock say?,get_time,0,clock say?,"[-0.6949534, 0.46386, -0.39600334, 0.13227333,..."
173,Launch WhatsApp chat window,open_whatsapp,5,Launch WhatsApp chat window,"[-0.781505, 0.016360007, -0.119255, 0.29608977..."
73,Find news channels on YouTube,search_youtube,2,Find news channels YouTube,"[-0.67197, 0.2058725, -0.3355865, 0.02870325, ..."


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.command_vector.values,
    df.intent_number,
    test_size=0.2,
    random_state=42
)

In [12]:
X_test.shape

(60,)

In [13]:
import numpy as np
X_train_2d = np.stack(X_train)
X_test_2d = np.stack(X_test)

In [14]:
X_train_2d

array([[-0.73865336,  0.06011667, -0.32663   , ..., -0.25181735,
         0.00284433,  0.48382   ],
       [-0.6662375 ,  0.44828498, -0.188895  , ...,  0.0629225 ,
         0.56246823,  0.0973785 ],
       [-0.73351   ,  0.41392   , -0.4425    , ..., -0.18777   ,
        -0.076822  , -0.015507  ],
       ...,
       [-0.72246504,  0.06410499, -0.125885  , ...,  0.09367501,
         0.142739  , -0.3148835 ],
       [-0.60877   ,  0.30253   , -0.12351   , ..., -0.096473  ,
        -0.46361   , -0.090981  ],
       [-0.7739067 ,  0.17369668, -0.04805334, ..., -0.01860233,
         0.30334333, -0.05477601]], shape=(240, 300), dtype=float32)

In [15]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report



scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_2d)
X_test_scaled = scaler.transform(X_test_2d)

base_models = [
    ('lr', LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='auto')),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('svm', SVC(kernel='linear', probability=True, random_state=42))
]

stack_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

stack_clf.fit(X_train_scaled, y_train)

y_pred = stack_clf.predict(X_test_scaled)
print("✅ Accuracy:", accuracy_score(y_test, y_pred)*100)
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))

joblib.dump(stack_clf, "jarvis_intent_model_stack.pkl")
joblib.dump(scaler, "jarvis_scaler_stack.pkl")
print("✅ Stacking model and scaler saved as 'jarvis_intent_model_stack.pkl' and 'jarvis_scaler_stack.pkl'")


✅ Accuracy: 98.33333333333333

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.86      1.00      0.92         6
           1       1.00      1.00      1.00         5
           2       1.00      1.00      1.00         7
           3       1.00      1.00      1.00         9
           4       1.00      1.00      1.00         2
           5       1.00      1.00      1.00         5
           6       1.00      1.00      1.00         5
           7       1.00      0.92      0.96        12
           8       1.00      1.00      1.00         3
           9       1.00      1.00      1.00         6

    accuracy                           0.98        60
   macro avg       0.99      0.99      0.99        60
weighted avg       0.99      0.98      0.98        60

✅ Stacking model and scaler saved as 'jarvis_intent_model_stack.pkl' and 'jarvis_scaler_stack.pkl'


In [17]:
import joblib
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import numpy as np

# 1️⃣ Load saved model and scaler
stack_clf = joblib.load("jarvis_intent_model_stack.pkl")
scaler = joblib.load("jarvis_scaler_stack.pkl")

# 2️⃣ Load spaCy model
nlp = spacy.load('en_core_web_md')

# 3️⃣ Define stopwords
stopwords = list(STOP_WORDS)

# 4️⃣ Map intent numbers back to intent names
intent_map = {
    0: 'get_time',
    1: 'search_google',
    2: 'search_youtube',
    3: 'open_notepad',
    4: 'open_calculator',
    5: 'open_whatsapp',
    6: 'open_linkedin',
    7: 'open_github',
    8: 'open_spotify',
    9: 'exit'
}

# 5️⃣ Function to preprocess and predict intent
def predict_intent(command):
    # Remove stopwords
    command_clean = ' '.join([word for word in command.split() if word.lower() not in stopwords])
    
    # Convert to embedding
    command_vector = nlp(command_clean).vector.reshape(1, -1)
    
    # Scale vector
    command_scaled = scaler.transform(command_vector)
    
    # Predict intent number
    intent_number = stack_clf.predict(command_scaled)[0]
    
    # Map number to intent name
    return intent_map[intent_number]

# 6️⃣ Test commands
test_commands = [
    "Open Spotify",
    "What time is it?",
    "Launch Notepad",
    "Search Python tutorials on Google",
    "Open my GitHub profile",
    "Exit Jarvis",
    "Tell me the current time",
    "Close the assistant"
]

# 7️⃣ Run predictions
for cmd in test_commands:
    intent_name = predict_intent(cmd)
    print(f"Command: '{cmd}' -> Predicted Intent: {intent_name}")
    print (accuracy_score)


Command: 'Open Spotify' -> Predicted Intent: open_spotify
<function accuracy_score at 0x0000023EB0CA4670>
Command: 'What time is it?' -> Predicted Intent: get_time
<function accuracy_score at 0x0000023EB0CA4670>
Command: 'Launch Notepad' -> Predicted Intent: open_notepad
<function accuracy_score at 0x0000023EB0CA4670>
Command: 'Search Python tutorials on Google' -> Predicted Intent: search_google
<function accuracy_score at 0x0000023EB0CA4670>
Command: 'Open my GitHub profile' -> Predicted Intent: open_github
<function accuracy_score at 0x0000023EB0CA4670>
Command: 'Exit Jarvis' -> Predicted Intent: exit
<function accuracy_score at 0x0000023EB0CA4670>
Command: 'Tell me the current time' -> Predicted Intent: get_time
<function accuracy_score at 0x0000023EB0CA4670>
Command: 'Close the assistant' -> Predicted Intent: exit
<function accuracy_score at 0x0000023EB0CA4670>
